# REINFORCE / Policy-Gradient Theorem — Williams 1992; Sutton, McAllester, Singh, Mansour 2000

_Papers · Reinforcement Learning_

**Optimize a policy DIRECTLY by gradient ascent on expected return: the estimator ∇J = E[∇ log π(a|s) · Gₜ], with an optional baseline that cuts the variance.**

---

This notebook is a practice scaffold for the **REINFORCE / Policy-Gradient Theorem — Williams 1992; Sutton, McAllester, Singh, Mansour 2000** lesson. We build it up one step at a time: the worked policy-gradient term by hand, the policy network, the discounted-return helper, and finally the training loop with and without a baseline. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (Colab)

### Step 1 — Check the worked policy-gradient term by hand

REINFORCE updates a policy by ascending `log π(a|s) · Gₜ`: weight the log-probability of each action taken by the **return** that followed it. An optional **baseline** `b` is subtracted from the return — it doesn't change the gradient in expectation, but it recenters the weight and cuts its variance.

We first reproduce the lesson's worked example by hand: a 3-step episode of all-1 rewards, with and without a baseline, to see how the loss term shrinks once `b` is subtracted.

In [ ]:
import math
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gymnasium as gym

torch.manual_seed(0)

# --- Sanity-check the lesson's worked example. ---
gamma = 0.99
G1 = 1 + gamma * 1 + gamma ** 2 * 1     # return that follows the first action
logp = math.log(0.6)                     # log-prob of the taken action (pi = 0.6)

loss_no_b = -logp * G1                    # REINFORCE loss term, no baseline

b = 2.0
loss_b = -logp * (G1 - b)                 # with running-mean baseline b = 2.0

print("worked example:  G1 =", round(G1, 4), " log(0.6) =", round(logp, 4),
      " -log(pi)*G1 =", round(loss_no_b, 4),
      " -log(pi)*(G1-b) =", round(loss_b, 4))
# worked example:  G1 = 2.9701  log(0.6) = -0.5108  -log(pi)*G1 = 1.5172  -log(pi)*(G1-b) = 0.4956

### Step 2 — Define the policy network and the return helper

The **policy** is a small network mapping a state to action logits, wrapped in a `Categorical` distribution so we can both sample an action and read its log-probability. The **discounted return** `Gₜ = Σ γᵏ r_{t+k}` is computed *backward* through the episode with the recurrence `G ← r + γ·G`, which is the cheap one-pass way to get every `Gₜ` at once.

In [ ]:
# --- Policy network (Track B: nn.Linear primitives). ---
class Policy(nn.Module):
    def __init__(self, obs_dim, n_act, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, n_act),        # -> action logits
        )
    def forward(self, x):
        return Categorical(logits=self.net(x))   # pi(a|s)


# --- Discounted return, computed BACKWARD: G <- r + gamma G  (G_t = sum_k gamma^k r_{t+k}). ---
def discounted_returns(rewards, gamma=0.99):
    G = 0.0
    out = [0.0] * len(rewards)
    for t in reversed(range(len(rewards))):
        G = rewards[t] + gamma * G
        out[t] = G
    return torch.tensor(out, dtype=torch.float32)

### Step 3 — Train REINFORCE on CartPole, with and without a baseline

The loop collects one on-policy episode, computes the returns, forms the **advantage** (return minus baseline), normalizes it for stability, and takes one gradient-ascent step on `log π · advantage`. The baseline here is a running mean of episode returns — it doesn't depend on the action, so it's a valid variance-reducing offset (Williams' `r − b`).

We run it twice: once with the baseline and once without (raw return). Both make the return rise, but the baselined run is smoother and reaches the solved score faster.

In [ ]:
# baseline=True -> subtract a running mean of returns (Williams' (r - b)); False -> raw return.
def train(use_baseline=True, episodes=600, gamma=0.99):
    torch.manual_seed(0)
    env = gym.make("CartPole-v1")
    pi = Policy(env.observation_space.shape[0], env.action_space.n)
    opt = torch.optim.Adam(pi.parameters(), lr=1e-2)

    running_b = 0.0
    recent = []
    history = []
    for ep in range(episodes):
        obs, _ = env.reset(seed=ep)
        logps, rews = [], []
        done = False
        while not done:                      # collect ONE on-policy episode
            dist = pi(torch.as_tensor(obs, dtype=torch.float32))
            a = dist.sample()
            logps.append(dist.log_prob(a))
            obs, r, term, trunc, _ = env.step(int(a))
            rews.append(float(r))
            done = term or trunc

        G = discounted_returns(rews, gamma)
        ep_ret = float(sum(rews))

        # Baseline = running mean of episode returns (does NOT depend on the action).
        if use_baseline:
            running_b = 0.95 * running_b + 0.05 * G.mean().item()
            adv = G - running_b
        else:
            adv = G - 0.0

        adv = (adv - adv.mean()) / (adv.std() + 1e-8)   # normalize for stability

        logp = torch.stack(logps)
        loss = -(logp * adv.detach()).sum()             # REINFORCE: ascent on logp*adv
        opt.zero_grad()
        loss.backward()
        opt.step()

        recent.append(ep_ret)
        recent = recent[-50:]
        avg = sum(recent) / len(recent)
        history.append(avg)
        if ep % 50 == 0 or avg >= 475:
            print(f"  ep {ep:3d}  avg return (last 50): {avg:6.1f}")
        if avg >= 475:
            print("  -> SOLVED CartPole.")
            break
    env.close()
    return history

print("REINFORCE with baseline (G - b):")
b_hist = train(use_baseline=True)

print("\nABLATION -- no baseline (raw return G, same everything else):")
raw_hist = train(use_baseline=False)

print("\nWith-baseline avg-return trajectory:", [round(h, 1) for h in b_hist[::40]])
print("No-baseline avg-return trajectory:  ", [round(h, 1) for h in raw_hist[::40]])
# The return rises on CartPole for both; the with-baseline run is smoother and reaches the
# solved score faster, the no-baseline ablation is noisier. (Exact numbers vary by
# hardware/seed; our small run, NOT a number reported by Williams 1992 or Sutton et al. 2000,
# which report theory results, not CartPole scores.)

## Visualize the data & results

_Does the bare REINFORCE estimator ∇J = E[∇log π(a|s)·Gₜ] make the CartPole return rise, and does subtracting a baseline (G − b, same everything else) make learning smoother / faster than the raw-return ablation? We train both for the same number of episodes and plot the average return._

### Step 1 — How the two curves above are produced

The visualization reuses the two `train` runs from Step 3 — there is no new training here. This cell summarizes how the with-baseline (`b_hist`) and no-baseline (`raw_hist`) average-return curves come out of the same loop, differing only in whether the advantage is `G − b` or the raw `G`. The single shared update is `loss = -(logp * adv.detach()).sum()`.

In [ ]:
# Sketch of how the two curves above are produced (full loop in the CODE cell).
# Train REINFORCE with a running-mean baseline and the no-baseline ablation on CartPole-v1
# for the same number of episodes with identical net / returns / lr / normalization / seed,
# recording the avg return (last 50 episodes).
#
#   b_hist   = train(use_baseline=True)    # green: adv = G - b; smooth climb to ~500 (SOLVED)
#   raw_hist = train(use_baseline=False)   # red:   raw G, no baseline; noisier, slower
#
# The key update is the same for both:
#   loss = -(logp * adv.detach()).sum()    # REINFORCE: gradient ascent on log(pi)*advantage
# with adv = G - b (green) vs adv = G (red).
#
# With baseline -> smoother monotone climb past the 475 "solved" line.
# No baseline   -> rises but jitters: the raw return's high variance makes each policy-
# gradient step noisy, so learning is slower and less stable.
# (Numbers are from our own run; neither paper reports a CartPole score.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked policy-gradient term. With $\gamma = 0.99$, a 3-step CartPole episode gives
            rewards $r_1 = r_2 = r_3 = 1$. The policy gave the first action probability $\pi = 0.6$. Compute
            the return $G_1$, the no-baseline loss term $-\log\pi\cdot G_1$, and the with-baseline loss term
            using $b = 2.0$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Return: $G_1 = 1 + 0.99(1) + 0.99^2(1) = 1 + 0.99 + 0.9801 = 2.9701$. — _The return that follows $a_1$ is the discounted sum of all rewards from step 1 onward, $G_t = \sum_{k\ge 0}\gamma^k r_{t+k}$._
- Log-prob: $\log\pi = \log 0.6 = -0.5108$. — _The policy gradient uses the log-probability of the action actually taken (the score function / characteristic eligibility)._
- No-baseline loss term: $-\log\pi\cdot G_1 = -(-0.5108)(2.9701) = +1.5172$. — _We minimize the negative of the objective $\log\pi\cdot G$; descending it raises $\log\pi$ because $G_1\gt 0$, pushing the action up._
- With baseline $b=2.0$: advantage $= 2.9701 - 2.0 = 0.9701$, loss term $= -(-0.5108)(0.9701) = +0.4956$. — _Subtracting the baseline recenters the weight around the average return — same sign, smaller magnitude, lower variance._

**Answer:** $G_1 = 2.9701$; no-baseline loss term $= +1.5172$; with-baseline ($b=2.0$) loss term
                 $= +0.4956$. Both are positive, so both push action $a_1$'s probability up — but the baseline
                 shrinks and recenters the update. The notebook recomputes $1 + 0.99 + 0.99^2 = 2.9701$,
                 $-\log(0.6)\cdot 2.9701 = 1.5172$, and $-\log(0.6)\cdot 0.9701 = 0.4956$.

</details>

**Problem 2.** The ablation. Your REINFORCE agent learns CartPole with a running-mean baseline. Remove the
            baseline (set $b = 0$, use the raw return $G_t$) — keeping the network, returns, learning rate, and
            seed identical — and retrain. What happens to the return curve, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the baseline: use -(logp * G.detach()).sum() instead of -(logp * (G - b).detach()).sum(); keep everything else fixed. — _An honest ablation changes exactly one thing — the baseline — so any difference is attributable to it._
- Retrain and watch the return: the no-baseline run is noisier and rises more slowly / less reliably than the baseline run. — _The raw return $G_t$ has high variance across episodes; the policy gradient weighted by it is noisy, so updates jitter. Subtracting $b$ recenters the signal and shrinks its variance._
- Conclude the baseline, not the network, is what makes learning steadier. — _Both runs share architecture, returns, and seed; only the baselined one is smooth, isolating the baseline as the cause._

**Answer:** The no-baseline agent learns more slowly and with a noisier, more jagged return curve, while
                 the baseline agent climbs more smoothly. Since the only difference is subtracting $b$, this
                 isolates the baseline as the variance reducer — exactly Williams' $(r - b)$ offset, and
                 the reason every later policy-gradient method (A2C, PPO) keeps a baseline/advantage. The
                 CODEVIZ panel shows this contrast.

</details>

**Problem 3.** Suppose at some step the return came out below the baseline: $G_t = 1.5$, $b = 2.0$, for an
            action with probability $\pi = 0.7$ ($\log\pi = -0.3567$). What is the loss term
            $-\log\pi\cdot(G_t - b)$, and which way does the update move this action's probability?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Advantage: $G_t - b = 1.5 - 2.0 = -0.5$. — _The return fell short of the average, so the advantage is negative — the action did worse than typical._
- Loss term: $-\log\pi\cdot(G_t - b) = -(-0.3567)(-0.5) = -0.1784$. — _Two negatives in $-\log\pi$ and one in the advantage give a negative product._
- Minimizing a negative loss term means the optimizer lowers it further by lowering $\log\pi$. — _Since the advantage is negative, the gradient of $-\log\pi\cdot(G-b)$ pushes $\log\pi$ down — the opposite of the positive-advantage case._

**Answer:** The loss term is $-0.1784$, and because the advantage is negative the update lowers
                 this action's probability — the policy learns to take it less often. This is the symmetry the
                 baseline gives: returns above the baseline push their action up, returns below push it down,
                 with the magnitude set by how far the return beat or missed the baseline.

</details>